<a href="https://colab.research.google.com/github/kk7188048/MLpoject/blob/main/Residual_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torch.utils.data.sampler import SubsetRandomSampler

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [ ]:
def train_valid_data():
  normalize = transforms.Normalize(
      mean=[0.4914, 0.4822, 0.4465],
      std=[0.2023, 0.1994, 0.2010],
  )
  tranform = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      normalize
  ])
  dataset = datasets.CIFAR10(
      root='./data',
      train=True,
      download=True,
      transform=tranform
  )
  num = len(dataset)
  indices = list(range(num))
  split = int(np.floor(0.1 * num))
  np.random.seed(10)
  np.random.shuffle(indices)
  train_idx, valid_idx = indices[split:], indices[:split]
  train_sampler = SubsetRandomSampler(train_idx)
  valid_sampler = SubsetRandomSampler(valid_idx)
  train = torch.utils.data.DataLoader(
      dataset, batch_size=64, sampler=train_sampler
  )
  valid = torch.utils.data.DataLoader(
      dataset, batch_size=64, sampler=valid_sampler
  )
  return train, valid


def test_data():
  normalize = transforms.Normalize(
      mean=[0.4914, 0.4822, 0.4465],
      std=[0.2023, 0.1994, 0.2010],
  )
  transform = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      normalize
  ])

  dataset = datasets.CIFAR10(
      root='./data',
      train=False,
      download=True,
      transform=transform
  )
  test = torch.utils.data.DataLoader(
      dataset, batch_size=64 ,shuffle=True
  )
  return test

train_data, valid_data = train_valid_data()
test_data = test_data()

train_images, train_labels = next(iter(train_data))
print("Shape of training images batch:", train_images.shape)
print("Shape of training labels batch:", train_labels.shape)

valid_images, valid_labels = next(iter(valid_data))
print("Shape of validation images batch:", valid_images.shape)
print("Shape of validation labels batch:", valid_labels.shape)

100%|██████████| 170M/170M [00:03<00:00, 43.7MB/s]


Shape of training images batch: torch.Size([64, 3, 224, 224])
Shape of training labels batch: torch.Size([64])
Shape of validation images batch: torch.Size([64, 3, 224, 224])
Shape of validation labels batch: torch.Size([64])


In [ ]:
class ResidualNetwork(nn.Module):
  def __init__(self, inside, outside, stride = 1, downsample=None) -> None:
    super().__init__()
    self.conv1 = nn.Sequential(
        nn.Conv2d(inside, outside, kernel_size=3, stride=stride, padding=1),
        nn.BatchNorm2d(outside),
        nn.ReLU()
    )
    self.conv2 = nn.Sequential(
        nn.Conv2d(outside, outside, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(outside)
    )
    self.downsampple = downsample
    self.relu = nn.ReLU()
    self.outside = outside

  def forward(self, x):
    residual = x
    out = self.conv1(x)
    out = self.conv2(out)
    if self.downsampple is not None:
      residual = self.downsampple(x)
    out += residual
    out = self.relu(out)
    return out




In [ ]:
print("""
Input
↓
Conv1 + MaxPool
↓
[Layer1]
 ├─ ResidualBlock1
 └─ ResidualBlock2
↓
[Layer2]
 ├─ ResidualBlock3 (downsample)
 └─ ResidualBlock4
↓
[Layer3]
 ├─ ResidualBlock5 (downsample)
 └─ ResidualBlock6
↓
[Layer4]
 ├─ ResidualBlock7 (downsample)
 └─ ResidualBlock8
↓
AvgPool → FC → Output
""")



Input
↓
Conv1 + MaxPool
↓
[Layer1]
 ├─ ResidualBlock1
 └─ ResidualBlock2
↓
[Layer2]
 ├─ ResidualBlock3 (downsample)
 └─ ResidualBlock4
↓
[Layer3]
 ├─ ResidualBlock5 (downsample)
 └─ ResidualBlock6
↓
[Layer4]
 ├─ ResidualBlock7 (downsample)
 └─ ResidualBlock8
↓
AvgPool → FC → Output



In [ ]:
class Resnet(nn.Module):
  def __init__(self, block , layers, num_class=10) -> None:
    super().__init__()
    self.inplanes = 64
    self.conv1 = nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
        nn.BatchNorm2d(64),
        nn.ReLU(),
    )
    self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    self.layer0 = self.make_layer(block,64, layers[0], stride=1)
    self.layer1 = self.make_layer(block,128, layers[1], stride=2)
    self.layer2 = self.make_layer(block,256, layers[2], stride=2)
    self.layer3 = self.make_layer(block, 512, layers[3], stride=2)
    self.avgpool = nn.AvgPool2d(7, stride=1)
    self.fc = nn.Linear(512, num_class)

  def make_layer(self, block, planes, blocks, stride=1):
    downsample = None
    if stride != 1 or self.inplanes != planes:
      downsample = nn.Sequential(
          nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride),
          nn.BatchNorm2d(planes)
      )
    layers = []
    layers.append(block(self.inplanes, planes, stride, downsample))
    self.inplanes = planes
    for i in range(1, blocks):
      layers.append(block(self.inplanes, planes))
    return nn.Sequential(*layers)

  def forward(self, x):
    x = self.conv1(x)
    x = self.maxpool(x)
    x = self.layer0(x)
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.avgpool(x)
    x = x.view(x.size(0), -1)
    x = self.fc(x)
    return x



In [ ]:
num_classes = 10
num_epochs = 20
learning_rate = 0.01
batch_size = 16

model = Resnet(ResidualNetwork, [3, 4, 6, 3]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay = 0.001, momentum = 0.9)
total_step = len(train_data)


In [ ]:
import gc
for epoch in range(num_epochs):
  for i, (images, labels) in enumerate(train_data):
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    del images, labels, outputs
    torch.cuda.empty_cache()
    gc.collect()

print ('Epoch [{}/{}], Loss: {:.4f}'
                       .format(epoch+1, num_epochs, loss.item()))

with torch.no_grad():
  correct = 0
  total = 0
  for images, labels in valid_data:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
    del images, labels, outputs
    torch.cuda.empty_cache()
    gc.collect()

  print('Accuracy of the network on the {} validation images: {} %'.format(5000, 100 * correct / total))



In [ ]:
with torch.no_grad():
  correct = 0
  total = 0
  for images, labes in test_data:
    images = images.to(device)
    labels = labels.to(device)
    output = model(images)
    _, predicted = torch.max(outputs.data, 1)
    total = labels.size(0)
    correct = (predicted == labels).sum().item()
    del images, labels, outputs
    torch.cuda.empty_cache()
    gc.collect()
    print('Accuracy of the network on the {} test images: {} %'.format(10000, 100 * correct / total))
